<a href="https://colab.research.google.com/github/devpalavicini/pouso_modulos_espacial/blob/main/Fase2Projeto2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""MGPEB - Missão Aurora Siger.

Combustível em porcentagem; massa em toneladas; prioridade 1 vem primeiro.
Cada rodada começa depois da chegada de todos os módulos à órbita.
O horário organiza a fila inicial; prioridade e combustível definem o atendimento.
Cada pouso autorizado é considerado concluído antes da próxima avaliação.
Não há simulação de passagem do tempo, consumo de combustível ou trajetória.

Regras do relatório:
    C = combustível >= 15%; A = clima adequado; D = pista livre;
    S = sensores íntegros; E = combustível < 5%.
    Pnominal = C and A and D and S
    Pemergencia = E and D
    Pautorizado = Pnominal or Pemergencia

A emergência acima ignora clima e sensores, inclusive com combustível zero.
Essa é uma regra acadêmica do relatório, não uma política de voo real.
"""

lista_modulos = [
    {
        "id_modulo": "MOD1",
        "tipo": "habitação",
        "prioridade": 1,
        "combustivel": 19.0,
        "massa_ton": 18.5,
        "criticidade_carga": "Alta",
        "horario_orbita": "14:30",
        "sensores_ok": True
    },
    {
        "id_modulo": "MOD2",
        "tipo": "energia",
        "prioridade": 1,
        "combustivel": 12.1,
        "massa_ton": 14.2,
        "criticidade_carga": "Alta",
        "horario_orbita": "14:35",
        "sensores_ok": True
    },
    {
        "id_modulo": "MOD3",
        "tipo": "suporte_medico",
        "prioridade": 2,
        "combustivel": 22.0,
        "massa_ton": 9.8,
        "criticidade_carga": "Alta",
        "horario_orbita": "14:40",
        "sensores_ok": True
    },
    {
        "id_modulo": "MOD4",
        "tipo": "laboratorio",
        "prioridade": 4,
        "combustivel": 18.9,
        "massa_ton": 12.0,
        "criticidade_carga": "Média",
        "horario_orbita": "14:45",
        "sensores_ok": True
    },
    {
        "id_modulo": "MOD5",
        "tipo": "logistica",
        "prioridade": 3,
        "combustivel": 20.0,
        "massa_ton": 22.0,
        "criticidade_carga": "Média",
        "horario_orbita": "14:50",
        "sensores_ok": True
    },
    {
        "id_modulo": "MOD6",
        "tipo": "mineração",
        "prioridade": 5,
        "combustivel": 7.1,
        "massa_ton": 25.1,
        "criticidade_carga": "Baixa",
        "horario_orbita": "14:55",
        "sensores_ok": True
    }
]


def validar_modulo(modulo):
    """Verifica o cadastro antes de usar os valores nas decisões."""
    if not isinstance(modulo, dict):
        raise ValueError("Cada módulo deve ser um dicionário.")

    campos = ["id_modulo", "tipo", "prioridade", "combustivel", "massa_ton",
              "criticidade_carga", "horario_orbita", "sensores_ok"]
    for campo in campos:
        if campo not in modulo:
            raise ValueError(f"Campo obrigatório ausente: {campo}.")

    for campo in ["id_modulo", "tipo", "criticidade_carga", "horario_orbita"]:
        if not isinstance(modulo[campo], str) or not modulo[campo].strip():
            raise ValueError(f"{campo} deve ser um texto não vazio.")

    if type(modulo["prioridade"]) is not int or modulo["prioridade"] < 1:
        raise ValueError("Prioridade deve ser um inteiro positivo.")

    combustivel = modulo["combustivel"]
    if type(combustivel) not in (int, float) or not 0 <= combustivel <= 100:
        raise ValueError("Combustível deve ser um número entre 0 e 100%.")

    massa = modulo["massa_ton"]
    if type(massa) not in (int, float) or not 0 < massa < float("inf"):
        raise ValueError("Massa deve ser um número positivo e finito.")

    if type(modulo["sensores_ok"]) is not bool:
        raise ValueError("sensores_ok deve ser True ou False.")

    horario = modulo["horario_orbita"]
    if len(horario) != 5 or horario[2] != ":":
        raise ValueError("Horário deve seguir o formato HH:MM.")
    for caractere in horario[:2] + horario[3:]:
        if caractere not in "0123456789":
            raise ValueError("Horário deve conter apenas números e dois-pontos.")
    if not (0 <= int(horario[:2]) <= 23 and 0 <= int(horario[3:]) <= 59):
        raise ValueError("Horário fora do intervalo de 00:00 a 23:59.")


def buscar_menor_combustivel(lista):
    """Busca linear. Em caso de empate, mantém o primeiro encontrado."""
    if not lista:
        return None
    menor = lista[0]
    for modulo in lista:
        if modulo["combustivel"] < menor["combustivel"]:
            menor = modulo
    return menor


def buscar_maior_prioridade(lista):
    """Maior prioridade operacional corresponde ao menor número."""
    if not lista:
        return None
    maior = lista[0]
    for modulo in lista:
        if modulo["prioridade"] < maior["prioridade"]:
            maior = modulo
    return maior


def buscar_tipo_carga(lista, tipo):
    """Busca linear por tipo de módulo/carga; retorna todos os encontrados."""
    if not isinstance(tipo, str):
        raise ValueError("O tipo de carga deve ser um texto.")
    encontrados = []
    for modulo in lista:
        if modulo["tipo"].strip().lower() == tipo.strip().lower():
            encontrados.append(modulo)
    return encontrados


def ordenar_fila_pouso(fila, por_horario=False):
    """Bubble Sort: horário na entrada; prioridade e combustível no atendimento.

    Empates completos mantêm a ordem anterior. A lista é alterada no lugar.
    """
    n = len(fila)
    for i in range(n):
        for j in range(0, n - i - 1):
            trocar = False
            if por_horario:
                trocar = fila[j]["horario_orbita"] > fila[j + 1]["horario_orbita"]
            elif fila[j]["prioridade"] > fila[j + 1]["prioridade"]:
                trocar = True
            elif fila[j]["prioridade"] == fila[j + 1]["prioridade"]:
                if fila[j]["combustivel"] > fila[j + 1]["combustivel"]:
                    trocar = True
            if trocar:
                fila[j], fila[j + 1] = fila[j + 1], fila[j]


def avaliar_autorizacao_pouso(modulo, clima_favoravel, pista_livre):
    """Aplica exatamente as regras booleanas declaradas no relatório."""
    try:
        validar_modulo(modulo)
    except ValueError as erro:
        return False, f"Pouso bloqueado: cadastro inválido. {erro}"

    if type(clima_favoravel) is not bool or type(pista_livre) is not bool:
        return False, "Pouso bloqueado: clima e pista devem ser True ou False."

    combustivel_suficiente = modulo["combustivel"] >= 15.0  # C
    combustivel_critico = modulo["combustivel"] < 5.0       # E
    sensores_ok = modulo["sensores_ok"]                    # S

    pouso_nominal = (
        combustivel_suficiente and clima_favoravel and pista_livre and sensores_ok
    )
    pouso_emergencia = combustivel_critico and pista_livre
    pouso_autorizado = pouso_nominal or pouso_emergencia

    if pouso_nominal:
        return pouso_autorizado, "Pouso nominal autorizado."
    elif pouso_emergencia:
        return pouso_autorizado, "Pouso de emergência autorizado pela regra E AND D."
    else:
        motivos = []
        if not pista_livre:
            motivos.append("pista ocupada")
        if not clima_favoravel:
            motivos.append("clima desfavorável")
        if not sensores_ok:
            motivos.append("falha nos sensores")
        if not combustivel_suficiente:
            motivos.append("combustível abaixo do mínimo nominal de 15%")
        return pouso_autorizado, "Pouso adiado: " + "; ".join(motivos) + "."


def exibir_modulos(titulo, lista):
    print(f"\n{titulo}")
    if not lista:
        print("Nenhum módulo.")
    for m in lista:
        print(f"{m['id_modulo']} | {m['tipo']} | Prioridade: {m['prioridade']} | "
              f"Combustível: {m['combustivel']}% | Órbita: {m['horario_orbita']}")


def processar_alertas(pilha_emergencia, exibir=True):
    """Desempilha em LIFO e preserva os registros; não resolve a falha física."""
    alertas_processados = []
    while pilha_emergencia:
        alerta = pilha_emergencia.pop()  # Último a entrar, primeiro a sair.
        alertas_processados.append(alerta)
        if exibir:
            print(f"Registro de alerta - {alerta['id_modulo']}: {alerta['motivo']}")
    return alertas_processados


def simular_pousos(lista_modulos, clima_ok=True, pista_livre=True, exibir=True):
    """Avalia cada módulo uma vez, sem modificar o cadastro recebido.

    Pista ocupada gera espera. Combustível abaixo de 15%, clima desfavorável
    ou sensores falhos também geram alerta. Um módulo pode estar na base ou
    em espera e, simultaneamente, na lista de ocorrências de alerta da rodada.
    """
    if not isinstance(lista_modulos, list):
        raise ValueError("O cadastro deve ser uma lista de módulos.")
    if type(clima_ok) is not bool or type(pista_livre) is not bool:
        raise ValueError("Clima e pista devem ser True ou False.")

    fila_pouso = []
    lista_base = []
    lista_espera = []
    lista_alerta = []
    pilha_emergencia = []
    historico_decisoes = []
    ids_cadastrados = []

    for modulo in lista_modulos:
        validar_modulo(modulo)
        if modulo["id_modulo"] in ids_cadastrados:
            raise ValueError(f"ID repetido: {modulo['id_modulo']}.")
        ids_cadastrados.append(modulo["id_modulo"])
        fila_pouso.append(modulo.copy())  # Preserva os dados originais.

    ordenar_fila_pouso(fila_pouso, por_horario=True)
    if exibir:
        exibir_modulos("FILA INICIAL - ORDEM DE CHEGADA À ÓRBITA", fila_pouso)
        menor = buscar_menor_combustivel(fila_pouso)
        maior = buscar_maior_prioridade(fila_pouso)
        if menor is not None:
            print(f"\nBusca - menor combustível: {menor['id_modulo']}.")
            print(f"Busca - maior prioridade (primeiro no empate): {maior['id_modulo']}.")
        exibir_modulos("BUSCA POR TIPO DE CARGA: energia",
                       buscar_tipo_carga(fila_pouso, "energia"))

    ordenar_fila_pouso(fila_pouso)
    if exibir:
        exibir_modulos("FILA REORGANIZADA - PRIORIDADE E COMBUSTÍVEL", fila_pouso)
        print("\nPROCESSAMENTO DE AUTORIZAÇÃO DE POUSO")

    while fila_pouso:
        modulo_atual = fila_pouso.pop(0)  # FIFO após a reorganização.
        autorizado, motivo = avaliar_autorizacao_pouso(
            modulo_atual, clima_ok, pista_livre
        )
        if autorizado:
            modulo_atual["status"] = "pousado"
            lista_base.append(modulo_atual)
        else:
            modulo_atual["status"] = "em_espera"
            lista_espera.append(modulo_atual)
        modulo_atual["motivo"] = motivo
        historico_decisoes.append({"id_modulo": modulo_atual["id_modulo"],
                                   "autorizado": autorizado, "motivo": motivo})
        if exibir:
            print(f"{modulo_atual['id_modulo']}: {motivo}")

        # Alertas são registrados mesmo quando a emergência autoriza o pouso.
        sinais_alerta = []
        if modulo_atual["combustivel"] < 15.0:
            sinais_alerta.append("combustível abaixo do mínimo nominal")
        if not modulo_atual["sensores_ok"]:
            sinais_alerta.append("falha nos sensores")
        if not clima_ok:
            sinais_alerta.append("condições atmosféricas desfavoráveis")
        if sinais_alerta:
            lista_alerta.append(modulo_atual)
            pilha_emergencia.append({"id_modulo": modulo_atual["id_modulo"],
                                     "motivo": "; ".join(sinais_alerta)})

    if exibir:
        print("\nLEITURA E REGISTRO DOS ALERTAS - LIFO")
    alertas_processados = processar_alertas(pilha_emergencia, exibir)

    if exibir:
        exibir_modulos("MÓDULOS POUSADOS", lista_base)
        exibir_modulos("MÓDULOS EM ESPERA", lista_espera)
        exibir_modulos("MÓDULOS COM OCORRÊNCIAS DE ALERTA NA RODADA", lista_alerta)
        print(f"\nResumo: {len(lista_base)} pousados; {len(lista_espera)} em espera; "
              f"{len(lista_alerta)} com alertas.")
        print("A leitura da pilha não resolve os alertas; seus registros foram preservados.")

    return {"fila_pouso": fila_pouso, "lista_base": lista_base,
            "lista_espera": lista_espera, "lista_alerta": lista_alerta,
            "pilha_emergencia": pilha_emergencia,
            "alertas_processados": alertas_processados,
            "historico_decisoes": historico_decisoes}


def demonstrar_cenarios():
    """Exemplos independentes, sem alterar os seis módulos cadastrados."""
    print("\nEXEMPLOS COMPLEMENTARES DE AUTORIZAÇÃO")
    cenarios = [
        ("Pouso nominal", 20.0, True, True, True),
        ("Pista ocupada", 20.0, True, False, True),
        ("Clima desfavorável", 20.0, False, True, True),
        ("Falha nos sensores", 20.0, True, True, False),
        ("Combustível intermediário", 12.0, True, True, True),
        ("Emergência conforme o relatório", 3.0, False, True, True)
    ]
    for nome, combustivel, clima_ok, pista_livre, sensores_ok in cenarios:
        exemplo = lista_modulos[0].copy()
        exemplo["combustivel"] = combustivel
        exemplo["sensores_ok"] = sensores_ok
        autorizado, motivo = avaliar_autorizacao_pouso(exemplo, clima_ok, pista_livre)
        print(f"{nome}: autorizado={autorizado} | {motivo}")


if __name__ == "__main__":
    resultado = simular_pousos(lista_modulos, clima_ok=True, pista_livre=True)
    demonstrar_cenarios()



FILA INICIAL - ORDEM DE CHEGADA À ÓRBITA
MOD1 | habitação | Prioridade: 1 | Combustível: 18.5% | Órbita: 14:30
MOD2 | energia | Prioridade: 1 | Combustível: 14.2% | Órbita: 14:35
MOD3 | suporte_medico | Prioridade: 2 | Combustível: 15.0% | Órbita: 14:40
MOD4 | laboratorio | Prioridade: 4 | Combustível: 12.0% | Órbita: 14:45
MOD5 | logistica | Prioridade: 3 | Combustível: 22.0% | Órbita: 14:50
MOD6 | mineração | Prioridade: 5 | Combustível: 7.1% | Órbita: 14:55

Busca - menor combustível: MOD6.
Busca - maior prioridade (primeiro no empate): MOD1.

BUSCA POR TIPO DE CARGA: energia
MOD2 | energia | Prioridade: 1 | Combustível: 14.2% | Órbita: 14:35

FILA REORGANIZADA - PRIORIDADE E COMBUSTÍVEL
MOD2 | energia | Prioridade: 1 | Combustível: 14.2% | Órbita: 14:35
MOD1 | habitação | Prioridade: 1 | Combustível: 18.5% | Órbita: 14:30
MOD3 | suporte_medico | Prioridade: 2 | Combustível: 15.0% | Órbita: 14:40
MOD5 | logistica | Prioridade: 3 | Combustível: 22.0% | Órbita: 14:50
MOD4 | laborator